# LCEL(LangChain Expression Language)
https://reference.langchain.com/python/langchain_core/runnables/

https://reference.langchain.com/python/langchain_core/runnables/?h=runnablelambd#langchain_core.runnables.base.RunnableLambda
  
- LCEL(LangChain Expression Language)은 LangChain에서 체인을 선언적으로 구성할 수 있게 해주는 도메인 특화 언어다.  
- `|` 연산자를 사용해 프롬프트, 모델, 파서 등을 파이프라인처럼 연결한다.

**주요 특징**

- **선언적 문법**: Unix 파이프처럼 `chain = prompt | model | parser` 형태로 직관적이다.  
- **모듈성·유연성**: 프롬프트, LLM, 파서, 검색기, 메모리 등 컴포넌트를 자유롭게 조합할 수 있다.  
- **동기/비동기 지원**: 단일 코드로 동기식·비동기식 실행을 모두 처리할 수 있다.  
- **병렬 처리 최적화**: 병렬 실행 가능한 단계는 자동으로 병렬화해 지연 시간을 줄인다.  
- **고급 기능 기본 제공**:  
  - 스트리밍 출력으로 응답 속도를 향상시킨다.  
  - 실패 시 재시도와 폴백 경로를 설정할 수 있다.  m
  - 중간 결과에 접근해 디버깅이나 진행 상황 표시가 가능하다.

**LCEL의 주요 기능**

1. **스트리밍 지원**: 첫 토큰 도달 시간을 단축해 실시간성을 높인다.  
2. **비동기 지원**: asyncio 환경 등 다양한 실행 환경을 동일 코드로 지원한다.  
3. **병렬 실행 최적화**: 병렬화 가능한 단계는 자동으로 분리해 동시에 실행한다.  
4. **재시도·폴백 구성**: 오류 발생 시 지정 횟수만큼 재시도하거나 대체 경로를 실행한다.  
5. **중간 결과 접근**: 최종 출력 이전에 각 단계의 출력을 확인할 수 있다.

**기본 구성 요소**

- **Runnable**: LCEL의 모든 컴포넌트가 상속하는 기본 클래스다.  
- **Chain**: 여러 Runnable을 순차적으로 실행한다.  
- **RunnableMap**: 여러 Runnable을 병렬로 실행한다.  
- **RunnableSequence**: Runnable들의 시퀀스를 정의한다.  
- **RunnableLambda**: 파이썬 함수를 래핑해 Runnable로 만든다.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

In [2]:
from langchain_core.runnables import RunnableLambda

runnable = RunnableLambda(lambda x: len(x))
runnable.invoke('안녕하냐')

4

In [3]:
runnable.batch(['안녕하냐', 'ㅁㅇㄴㄻㅇㄹㄴㄹ'])

[4, 8]

In [4]:
def celsius_to_fahrenheit(celsius):
    return celsius * 9 / 5 + 32

celsius_temp = [0, 25, 100, -10, 36]
runnable = RunnableLambda(celsius_to_fahrenheit)
runnable.batch(celsius_temp)

[32.0, 77.0, 212.0, 14.0, 96.8]

In [8]:
import time

def generator(x):
    for y in x:
        yield y

runnable = RunnableLambda(generator)
for chunk in runnable.stream('안녕하세요😊😂🤣❤️😍😒안녕하세요😊😂🤣❤️😍😒'):
    print(chunk, end='', flush=True)
    time.sleep(0.1)

안녕하세요😊😂🤣❤️😍😒안녕하세요😊😂🤣❤️😍😒

In [10]:
def g(x):
    for y in x:
        yield y

gen10 = g(range(10))
gen10

<generator object g at 0x0000018E99EE19C0>

In [12]:
from langchain_core.runnables import RunnableParallel

runnable1 = RunnableLambda(lambda x: {'foo': x})
runnable2 = RunnableLambda(lambda x: {'r2': [x, x, x]})

chain = RunnableParallel(
    r1=runnable1,
    r2=runnable2
)

result = chain.invoke('hello')

print(result)

{'r1': {'foo': 'hello'}, 'r2': {'r2': ['hello', 'hello', 'hello']}}


In [13]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model


llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

acrostic_poem_prompt = PromptTemplate.from_template(
    '당신은 n행시 고수. 다음 주제로 n행시 생성. 주제: {topic}'
)

joke_prompt = PromptTemplate.from_template(
    '당신은 한국식 농담 고수. 다음 주제로 웃긴 농담 생성. 주제: {topic}'
)

poem_prompt = PromptTemplate.from_template(
    '당신은 현대 시 작가. 다음 주제로 감성적 시 생성. 주제: {topic}'
)

acrostic_chain = acrostic_poem_prompt | llm | output_parser
joke_chain = joke_prompt | llm | output_parser
poem_chain = poem_prompt | llm | output_parser

chain = RunnableParallel(
    acrostic_poem=acrostic_chain,
    joke=joke_chain,
    poem=poem_chain
)

def combine_result(input_dict: dict) -> str:
    acrostic_poem = input_dict['acrostic_poem']
    joke = input_dict['joke']
    poem = input_dict['poem']
    return f"""
    n행시
    {acrostic_poem}

    농담
    {joke}

    현대 시
    {poem}
    """

chain = chain | RunnableLambda(combine_result)
print(chain.invoke({'topic': 'IT'}))


    n행시
    **아**무리 복잡한 문제도  
**이**제는 기술로 척척 풀고  
**티**끌만 한 아이디어도 세상을 바꾸는 **IT!**

    농담
    개발자가 가장 싫어하는 계절은?

**버그가 많이 나는 가을…**  
왜냐하면 코드가 자꾸 **낙엽처럼 떨어지거든.**

    현대 시
    ### 서버실의 새벽

모니터의 푸른빛이  
잠들지 못한 얼굴을 비추고,

도시는 수억 개의 신호로  
서로의 안부를 묻는다.

어딘가에서 누군가  
작은 하트를 보내는 동안  
광섬유 속을 달리던 마음 하나가  
대륙을 건너  
당신의 손끝에 도착한다.

우리는 이제  
목소리보다 먼저 접속하고,  
눈빛보다 먼저 읽음 표시를 확인하며,  
사라진 사람의 마지막 흔적을  
클라우드에 저장한다.

하지만 아무리 정교한 알고리즘도  
그리움의 지연 시간을 계산하지 못하고,  
어떤 인공지능도  
침묵 속에 남은 이름 하나를  
완전히 해독하지 못한다.

새벽 두 시,  
서버 팬이 낮게 울고  
도시는 여전히 깨어 있다.

나는 로그아웃하지 못한 채  
당신의 오래된 메시지를 열어 본다.

“잘 지내?”

짧은 문장 하나가  
수많은 데이터보다 깊이  
내 안에 접속한다.
    


In [14]:
n_poem_prompt = PromptTemplate.from_template(
    '너 n행시 고수. 다음 주제로 n행시 작성. 주제: {topic}'
)
n_poem_chain = n_poem_prompt | llm | output_parser

print(n_poem_chain.invoke({'topic': 'AI'}))

**A**: 알아서 척척 답을 내놓지만  
**I**: 이 세상에 온기는 사람에게서 배운다


In [15]:
from langchain_core.runnables import RunnablePassthrough

prompt = PromptTemplate.from_template(
    '너 n행시 고수. 다음 주제로 n행시 작성. 주제: {topic}'
)
chain = {'topic': RunnablePassthrough()} | n_poem_prompt | llm | output_parser

print(n_poem_chain.invoke({'topic': 'AI'}))

**AI 2행시**

**에이:** 에이, 그건 사람만 할 수 있다고요?  
**아이:** 아이디어만 주세요, 현실로 만들어드릴게요!


In [16]:
prompt = PromptTemplate.from_template(
    """너 {n}행시 고수. 다음 주제로 {n}행시 작성. 주제: {topic}

출력 형식
===== <주제> <{n}행시> =====
<n행시 작성>
""")
chain = ({'topic': RunnablePassthrough()}
        | RunnablePassthrough.assign(
                n=lambda x: len(x['topic']),
                k=lambda x: 100
        )
        | prompt
        | llm
        | output_parser
)
print(chain.invoke({'topic': '아이스크림'}))

===== 아이스크림 1행시 =====  
아무리 추워도 아이스크림 앞에서는 내 마음이 먼저 녹는다.
